# Music by Prompt v2: encoder–decoder + LoRA

В Colab выбери **Среда выполнения → Сменить среду выполнения → GPU**.
Затем выполни все ячейки сверху вниз.

Один этап: LoRA во всех attention + теги/проекции обучаются вместе. Исходные Transformer-веса заморожены всегда.
Данные уже подготовлены: `MyDrive/music_prompt/data/features.npz` и `data/pilot/` целиком.
Архив кода: `twotower_lora.zip` (свежий, из этого изменения). GitHub не требуется.

50 000 train-диалогов; история 64; текст 128; E=2; cross через 4 слоя; rank=8.
20 первых шагов каждого режима входят в обучение. По скорости выбираются общие 2 или 4 эпохи.
Если даже 2 эпохи не помещаются, скрипт остановится после замера и сообщит об этом.


In [ ]:
from pathlib import Path
import zipfile
from google.colab import files

ROOT = Path('/content/music_prompt')
ROOT.mkdir(parents=True, exist_ok=True)
print('Выбери twotower_lora.zip с компьютера')
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith('.zip'))
with zipfile.ZipFile(archive) as z:
    z.extractall(ROOT)
assert (ROOT / 'twotower/experiment.py').is_file(), 'Нужен новый архив twotower_lora.zip'


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchvision', 'torchaudio'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(ROOT / 'twotower/requirements.txt')], check=True)


In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/music_prompt')
source = DRIVE_ROOT / 'data'
if not (source / 'features.npz').is_file():
    source = DRIVE_ROOT / 'data/model'  # поддерживается и прежняя раскладка
assert (source / 'features.npz').is_file(), f'Нет {source / "features.npz"}'
assert (source / 'pilot/manifest.json').is_file(), f'Нет {source / "pilot/manifest.json"}'
local = ROOT / 'data/model'
local.mkdir(parents=True, exist_ok=True)
shutil.copy2(source / 'features.npz', local / 'features.npz')
shutil.copytree(source / 'pilot', local / 'pilot', dirs_exist_ok=True)
print('Данные скопированы на локальный диск Colab:', local)


In [ ]:
import json

# Сохраняй это имя при продолжении. Для независимого нового эксперимента поменяй его.
RUN_NAME = 'lora_v2_e2'
RUN_DIR = DRIVE_ROOT / 'runs' / RUN_NAME
HOURS = 22  # бюджет этого запуска; оставляем время от суток на установку/копирование

# Импорт torch делаем в новом процессе после установки зависимостей.
gpu = subprocess.check_output([sys.executable, '-c',
    'import torch; assert torch.cuda.is_available(), "Включи GPU"; print(torch.cuda.get_device_name())'], text=True).strip()
BATCH = 32 if 'A100' in gpu else 8
# При продолжении после смены GPU сохраняем прежний batch для точного resume.
settings = RUN_DIR / 'experiment.json'
if settings.exists():
    BATCH = json.loads(settings.read_text())['batch_size']
print('GPU:', gpu, '| batch:', BATCH, '| результаты:', RUN_DIR, flush=True)
command = [sys.executable, '-u', '-m', 'twotower.experiment',
           '--out', str(RUN_DIR), '--hours', str(HOURS), '--batch-size', str(BATCH)]
# Вывод шагов появляется сразу. При обрыве снова выполни ячейки с тем же RUN_NAME.
with subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, bufsize=1) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
    status = process.wait()
if status:
    raise RuntimeError(f'Запуск остановлен с кодом {status}; причина выше')
result = RUN_DIR / 'results.json'
if result.exists():
    print(result.read_text())
else:
    print('Общий эксперимент ещё не завершён. Причина остановки указана выше; checkpoint на Drive.')


## Где результаты и как продолжить

`MyDrive/music_prompt/runs/lora_v2_e2/`:
- `plan.json`: выбранное число эпох и замер скорости;
- `both/`, `text/`, `history/`: `metrics.jsonl`, `config.json`, `last.pt`, `best.pt`;
- `*/evaluation.json`: итоговая оценка лучшего checkpoint на всех 8421 val-ходах;
- `baselines_all_val.json`: популярность и последние треки истории на том же val;
- `results.json`: общая таблица в JSON после завершения всего эксперимента.

`last.pt` сохраняется каждые 100 шагов и в конце этапа; `best.pt` — по NDCG@100.
При разрыве среды снова выполни ячейки с тем же RUN_NAME: обучение продолжится с last.pt.
До 99 последних несохранённых шагов могут повториться. Новый вызов даёт новый лимит HOURS,
не отменяет уже потраченное время. При смене A100 на T4 старый batch=32 может не поместиться;
для точного resume нужен совместимый GPU, автоматически batch не уменьшается.

Это новый эксперимент: старые fully-trained checkpoint к нему не подмешиваются.
Ни код, ни данные не отправляются на GitHub.
